In [1]:
from torch.utils.data import TensorDataset
import numpy as np
import torch
from tqdm import tqdm

from SpringSim import SpringSim
from nri.NRI import NRIModule

In [2]:
prior = np.array([0.5, 0, 0.5])

def create_dataset(size: int):
    sim = SpringSim()
    trajectories = []
    for _ in tqdm(range(size)):
        xs, vs, edge = sim.sample_trajectory(1000, 10, prior)
        node_features = np.concatenate([xs, vs], axis=1)
        trajectories.append(node_features.transpose(0, 2, 1))

    return TensorDataset(torch.Tensor(np.stack(trajectories, axis=0)))

ds_train = create_dataset(1000)
ds_test = create_dataset(100)

100%|██████████| 100/100 [00:03<00:00, 27.37it/s]


In [3]:
from nri.train_nri import train

nri_module = NRIModule(
    x_dim=4,
    hidden_dim=8,
    trajectory_length=99,
    num_edge_types=3,
    pred_steps=4,
    dropout_prob=0.1,
    skip_first=True
)
train(nri_module, ds_train, ds_test, prior, show_latent_edges_on_eval=False, tensorboard_logger=None, num_epochs=50)

/home/adrian/Dev/NRI-for-explainable-RL-in-Power-Grids/nri/utils.py:104: UserWarning: index_reduce() is in beta and the API may change at any time. (Triggered internally at /home/conda/feedstock_root/build_artifacts/libtorch_1739474892959/work/aten/src/ATen/native/TensorAdvancedIndexing.cpp:1218.)
  agg.index_reduce_(dim=-2, index=receivers, source=e, reduce="mean")


Epoch 0: Training loss: 12.28 Neg Log Likelihood: 5.02 KL Divergence: 7.26 MSE: 2.68
Epoch 0: Training loss: 11.30 Neg Log Likelihood: 5.62 KL Divergence: 5.67 MSE: 3.90
Epoch 1: Training loss: 8.31 Neg Log Likelihood: 4.59 KL Divergence: 3.72 MSE: 1.82
Epoch 2: Training loss: 7.03 Neg Log Likelihood: 4.38 KL Divergence: 2.64 MSE: 1.41
Epoch 3: Training loss: 6.26 Neg Log Likelihood: 4.24 KL Divergence: 2.01 MSE: 1.14
Epoch 4: Training loss: 5.77 Neg Log Likelihood: 4.13 KL Divergence: 1.64 MSE: 0.91
Epoch 5: Training loss: 5.44 Neg Log Likelihood: 4.04 KL Divergence: 1.40 MSE: 0.74
Epoch 6: Training loss: 5.21 Neg Log Likelihood: 3.98 KL Divergence: 1.23 MSE: 0.60
Epoch 7: Training loss: 5.03 Neg Log Likelihood: 3.92 KL Divergence: 1.11 MSE: 0.49
Epoch 8: Training loss: 4.90 Neg Log Likelihood: 3.88 KL Divergence: 1.02 MSE: 0.40
Epoch 9: Training loss: 4.79 Neg Log Likelihood: 3.84 KL Divergence: 0.95 MSE: 0.33
Epoch 10: Training loss: 4.77 Neg Log Likelihood: 3.83 KL Divergence: 0.94

NRIModule(
  (encoder): Encoder(
    (f_emb): MLP(
      (fc1): Linear(in_features=396, out_features=8, bias=True)
      (fc2): Linear(in_features=8, out_features=8, bias=True)
      (bn): BatchNorm1d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (node2edge_1): Node2Edge(
      (psi): MLP(
        (fc1): Linear(in_features=16, out_features=8, bias=True)
        (fc2): Linear(in_features=8, out_features=8, bias=True)
        (bn): BatchNorm1d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (edge2node): Edge2Node(
      (phi): MLP(
        (fc1): Linear(in_features=8, out_features=8, bias=True)
        (fc2): Linear(in_features=8, out_features=8, bias=True)
        (bn): BatchNorm1d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (node2edge_2): Node2Edge(
      (psi): MLP(
        (fc1): Linear(in_features=16, out_features=8, bias=True)
        (fc2): Linear(in_features=8, out_feature